# Speech Transit Toolformer demo

This local notebook calls project package code and CLI entry points. It keeps demo orchestration in notebook cells, while dataset generation, pipeline execution, and evaluation remain in `src/` and `scripts/`.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'configs').exists():
    raise RuntimeError('Run this notebook from the repository root.')

PYTHON = PROJECT_ROOT / '.venv' / 'bin' / 'python'
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

QUICK_DEMO = True
QUICK_N = 5
print(f'Project root: {PROJECT_ROOT}')
print(f'Python: {PYTHON}')
print(f'Quick demo mode: {QUICK_DEMO}')

## Load configs

Use `configs/fast_model.yaml` for faster text testing. Use `configs/reference_model.yaml` for reference and multimodal runs.

In [ ]:
from src.utils.config import load_yaml_config

dataset_config = load_yaml_config('configs/dataset.yaml')
pipeline_config = load_yaml_config('configs/pipelines.yaml')
evaluation_config = load_yaml_config('configs/evaluation.yaml')
fast_model_config = load_yaml_config('configs/fast_model.yaml')
reference_model_config = load_yaml_config('configs/reference_model.yaml')

print('Dataset examples:', dataset_config['generation']['total_examples'])
print('Fast text model:', fast_model_config['model']['id'])
print('Reference model:', reference_model_config['model']['id'])

## Generate or load text dataset

In [ ]:
text_test_path = PROJECT_ROOT / dataset_config['outputs']['test']
if not text_test_path.exists():
    subprocess.run([str(PYTHON), '-m', 'src.cli', 'generate-text-dataset'], cwd=PROJECT_ROOT, check=True)
else:
    print(f'Using existing text test split: {text_test_path}')

from src.data.loaders.jsonl import read_jsonl
test_rows = read_jsonl(text_test_path)
quick_rows = test_rows[:QUICK_N]
print(f'Test rows: {len(test_rows)}; quick rows: {len(quick_rows)}')

## Generate or load audio dataset

In [ ]:
audio_metadata_path = PROJECT_ROOT / dataset_config['outputs']['audio_metadata']
if not audio_metadata_path.exists():
    print('Audio metadata is missing. Run scripts/generate_audio_dataset.sh or the CLI when Piper voices are available.')
    print('Command:', f'{PYTHON} -m src.cli generate-audio-dataset')
else:
    print(f'Using existing audio metadata: {audio_metadata_path}')

## Run Pipeline A: text -> tool call

In [ ]:
if QUICK_DEMO:
    from src.data_models import DatasetExample
    from src.models.inference.text_model import StubTextBackend, TextModelInference
    from src.pipelines.pipeline_a.runner import run_pipeline_a
    from src.data.loaders.jsonl import write_jsonl

    quick_dir = PROJECT_ROOT / 'data' / 'demo'
    quick_dir.mkdir(parents=True, exist_ok=True)
    quick_text_path = quick_dir / 'quick_test.jsonl'
    write_jsonl(quick_text_path, quick_rows)
    responses = []
    for row in quick_rows:
        expected = row.get('expected_tool_call')
        responses.append(json.dumps({'tool_call': expected}, ensure_ascii=False) if expected else 'No transport lookup needed.')
    inference = TextModelInference(backend=StubTextBackend(responses, model_name='quick-stub-text'))
    pipeline_a_records = run_pipeline_a(input_path=quick_text_path, output_path=quick_dir / 'pipeline_a_predictions.jsonl', inference=inference)
else:
    subprocess.run([str(PYTHON), '-m', 'src.cli', 'run-pipeline-a', '--config', 'configs/fast_pipelines.yaml'], cwd=PROJECT_ROOT, check=True)
print('Pipeline A records:', len(pipeline_a_records) if QUICK_DEMO else 'written by CLI')

## Run Pipelines B, C, and D

Pipeline B is audio -> transcript, Pipeline C is audio -> transcript + tool call, and Pipeline D is audio -> transcript -> tool call. Quick mode uses the package runners with stub model backends; full mode should run the CLI against the fixed test split and `configs/reference_model.yaml`.

In [ ]:
if QUICK_DEMO:
    print('Quick local demo for B/C/D requires generated WAV files. Use the Colab notebook for a complete quick-mode path with audio guards.')
else:
    for command in ['run-pipeline-b', 'run-pipeline-c', 'run-pipeline-d']:
        subprocess.run([str(PYTHON), '-m', 'src.cli', command, '--config', 'configs/pipelines.yaml'], cwd=PROJECT_ROOT, check=True)

## Unified evaluation and metrics tables

In [ ]:
if not QUICK_DEMO:
    subprocess.run([str(PYTHON), '-m', 'src.cli', 'evaluate', '--config', 'configs/evaluation.yaml'], cwd=PROJECT_ROOT, check=True)

import pandas as pd
comparison_table = PROJECT_ROOT / evaluation_config['outputs']['comparison_table']
if comparison_table.exists():
    display(pd.read_csv(comparison_table))
else:
    print('Metrics table not found yet. Run all four pipelines and unified evaluation first.')

## Failure examples and pipeline choice

In [ ]:
failure_path = PROJECT_ROOT / evaluation_config['outputs']['failure_cases']
if failure_path.exists():
    failures = pd.read_json(failure_path, lines=True)
    display(failures.head(10))
else:
    print('Failure examples are produced by unified evaluation.')

if comparison_table.exists():
    metrics = pd.read_csv(comparison_table)
    candidates = metrics[metrics['metric'].eq('tool_exact_match_accuracy')].sort_values('value', ascending=False)
    display(candidates)
    if not candidates.empty:
        print('Best pipeline by tool exact-match accuracy:', candidates.iloc[0]['pipeline'])
else:
    print('Choose the best pipeline after comparing tool accuracy, ASR WER, and failure buckets.')